In [3]:
# ============================================================
# DNN training, validation, and external verification workflow
# On Jupyter Notebook
#

In [4]:
# ============================================================
# 1. Import required modules
# ============================================================

import random
import copy

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
# ============================================================
# 2. Basic settings
# ============================================================

# Please modify this path according to your local file location
data_path = r"path/Train_Validation_Verification_Data_revised.xlsx"

# Random seed used for model initialization and data shuffling
MODEL_SEED = 1

# Fixed model architecture
NUM_LAYERS = 4
HIDDEN_SIZE = 48

# Training settings
N_EPOCHS = 50
LR = 0.01
BATCH_SIZE = 256
PATIENCE = 10
MIN_DELTA = 1e-1
WARMUP_EPOCHS = 10
DROP_LAST = False

# Model parameter file generated from the current notebook run
MODEL_PARAM_SAVE_PATH = "DNN_para_final.pth"

In [6]:
# ============================================================
# 3. Reproducibility setting
# ============================================================

def set_seed(seed=42):
    # Set random seeds for reproducibility.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass


set_seed(MODEL_SEED)

In [7]:
# ============================================================
# 4. Load training, validation, and external verification data
# ============================================================

# Train sheet = training set
train_data = pd.read_excel(data_path, sheet_name="Train")

# Val sheet = validation set used to guide early stopping but not used for gradient-based parameter updating.
val_data = pd.read_excel(data_path, sheet_name="Val")

feature_train = train_data.iloc[:, :10].to_numpy(dtype=np.float32)
label_train = train_data.iloc[:, -1].to_numpy(dtype=np.float32)

feature_val = val_data.iloc[:, :10].to_numpy(dtype=np.float32)
label_val = val_data.iloc[:, -1].to_numpy(dtype=np.float32)

# External verification inputs
verification_features = pd.read_excel(
    data_path,
    sheet_name="Verification-inputs"
).iloc[:, :10].to_numpy(dtype=np.float32)

# Experimental residual-mass curves for the 15 external verification experiments
true_df = pd.read_excel(
    data_path,
    sheet_name="Verification experiments"
)

In [8]:
# ============================================================
# 5. Dataset and DNN model definition
# ============================================================

class ExcelDataset(Dataset):
    def __init__(self, features, labels):
        self.x = torch.from_numpy(features).float()
        self.y = torch.from_numpy(labels).float()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return {
            "x": self.x[index],
            "y": self.y[index]
        }


class BP(nn.Module):
    """
    DNN model used for residual-mass prediction.
    """

    def __init__(self, num_layers, hidden_size):
        super(BP, self).__init__()

        self.Liner1 = nn.Linear(10, hidden_size)

        self.queue = [
            nn.Sequential(
                nn.Linear(hidden_size, hidden_size),
                nn.LeakyReLU(0.1)
            )
            for _ in range(num_layers - 1)
        ]

        self.model = nn.Sequential(*self.queue)
        self.Liner2 = nn.Linear(hidden_size, 1)

    def forward(self, input):
        output = self.Liner1(input)
        output = self.model(output)
        output = self.Liner2(output)
        return output

In [9]:
# ============================================================
# 6. Unified prediction and metric functions
# ============================================================

def predict_array(model, features):
    """
    Generate raw DNN predictions.

    No smoothing, normalization, first-point correction, or other
    post-processing is applied.
    """
    model.eval()

    features = np.asarray(features, dtype=np.float32)

    with torch.no_grad():
        x = torch.from_numpy(features).float()
        pred = model(x).squeeze(1).detach().numpy()

    return pred.reshape(-1)


def calculate_metrics_from_arrays(y_true, y_pred):
    """
    Calculate evaluation metrics directly from raw predicted and experimental values.
    """
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)

    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    mask = mask & np.isfinite(y_true) & np.isfinite(y_pred)

    y_true_valid = y_true[mask]
    y_pred_valid = y_pred[mask]

    if len(y_true_valid) == 0:
        raise ValueError("No valid data points are available for metric calculation.")

    error = y_pred_valid - y_true_valid
    abs_error = np.abs(error)

    r2 = r2_score(y_true_valid, y_pred_valid)
    rmse = np.sqrt(mean_squared_error(y_true_valid, y_pred_valid))
    mae = mean_absolute_error(y_true_valid, y_pred_valid)
    mbe = np.mean(error)
    medae = np.median(abs_error)

    denominator = np.abs(y_true_valid) + np.abs(y_pred_valid)
    smape = 200 * np.mean(
        np.abs(y_pred_valid - y_true_valid) / np.maximum(denominator, 1e-8)
    )

    return {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "MBE": mbe,
        "MedAE": medae,
        "sMAPE(%)": smape,
        "N_points": len(y_true_valid)
    }


def evaluate_internal_dataset(model, features, labels):
    """
    Evaluate the training or validation dataset.
    """
    pred = predict_array(model, features)
    metrics = calculate_metrics_from_arrays(labels, pred)
    return metrics, pred


def evaluate_15_verification_experiments(model, verification_features, true_df):
    """
    Evaluate the 15 external practical MSW verification experiments.

    The metrics are calculated directly from:
    - raw DNN predictions
    - raw experimental residual-mass values

    No smoothing, first-point correction, normalization, or other post-processing
    is applied before metric calculation.
    """

    dnn_pred = predict_array(model, verification_features)

    group_size = true_df.shape[0]
    n_groups = true_df.shape[1]
    expected_length = group_size * n_groups

    if len(dnn_pred) != expected_length:
        raise ValueError(
            "The length of DNN predictions does not match the experimental data. "
            f"Prediction length = {len(dnn_pred)}, "
            f"expected length = {expected_length} "
            f"({n_groups} experiments × {group_size} points)."
        )

    pred_df_raw = pd.DataFrame({
        true_df.columns[i]: dnn_pred[i * group_size:(i + 1) * group_size]
        for i in range(n_groups)
    })

    true_df_raw = true_df.copy().astype(float)
    pred_df_raw = pred_df_raw.copy().astype(float)

    results = []

    for col in true_df_raw.columns:
        y_true = true_df_raw[col].to_numpy(dtype=float)
        y_pred = pred_df_raw[col].to_numpy(dtype=float)

        metrics = calculate_metrics_from_arrays(y_true, y_pred)

        results.append({
            "Experiment": col,
            "N_points": metrics["N_points"],
            "R2": metrics["R2"],
            "RMSE": metrics["RMSE"],
            "MAE": metrics["MAE"],
            "MBE": metrics["MBE"],
            "MedAE": metrics["MedAE"],
            "sMAPE(%)": metrics["sMAPE(%)"],
        })

    results_df = pd.DataFrame(results)

    summary_df = results_df[
        ["R2", "RMSE", "MAE", "MBE", "MedAE", "sMAPE(%)"]
    ].agg(["mean", "std", "min", "max", "median"]).T.reset_index()

    summary_df = summary_df.rename(columns={
        "index": "Metric",
        "mean": "Mean",
        "std": "Std",
        "min": "Min",
        "max": "Max",
        "median": "Median"
    })

    return results_df, summary_df, pred_df_raw


def print_metric_summary(name, metrics):
    """
    Print metrics in a compact format.
    """
    print(
        "%s | R2=%.5f | RMSE=%.4f | MAE=%.4f | MBE=%.4f | MedAE=%.4f | sMAPE=%.4f%%"
        % (
            name,
            metrics["R2"],
            metrics["RMSE"],
            metrics["MAE"],
            metrics["MBE"],
            metrics["MedAE"],
            metrics["sMAPE(%)"]
        )
    )

In [10]:
# ============================================================
# 7. DNN training with validation-based early stopping
# ============================================================

def train_with_validation_early_stopping(
    feature_train,
    label_train,
    feature_val,
    label_val,
    seed=1
):
    """
    Train the DNN model using the training set.

    The validation set is used to monitor validation loss and guide the
    early-stopping criterion. Validation data are not used for gradient-based
    parameter updating.

    The retained model parameters are generated during the current notebook run
    and saved to MODEL_PARAM_SAVE_PATH.
    """

    set_seed(seed)

    dataset = ExcelDataset(feature_train, label_train)

    data_generator = torch.Generator()
    data_generator.manual_seed(seed)

    dataloader = DataLoader(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        drop_last=DROP_LAST,
        generator=data_generator
    )

    model = BP(NUM_LAYERS, HIDDEN_SIZE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.MSELoss()

    retained_val_loss = np.inf
    retained_epoch = None
    wait = 0
    retained_model_state = None

    train_loss_history = []
    val_loss_history = []

    for epoch in range(N_EPOCHS):
        model.train()

        for batch in dataloader:
            x = batch["x"]
            y = batch["y"]

            pred = model(x).squeeze(1)
            loss = loss_fn(pred, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()

        with torch.no_grad():
            train_pred_epoch = predict_array(model, feature_train)
            val_pred_epoch = predict_array(model, feature_val)

        train_loss_eval = mean_squared_error(label_train, train_pred_epoch)
        val_loss_eval = mean_squared_error(label_val, val_pred_epoch)

        train_loss_history.append(train_loss_eval)
        val_loss_history.append(val_loss_eval)

        train_r2 = r2_score(label_train, train_pred_epoch)
        val_r2 = r2_score(label_val, val_pred_epoch)

        print(
            "Epoch %03d/%03d | Train_loss=%.6f | Val_loss=%.6f | Train_R2=%.5f | Val_R2=%.5f"
            % (
                epoch + 1,
                N_EPOCHS,
                train_loss_eval,
                val_loss_eval,
                train_r2,
                val_r2
            )
        )

        # Validation-based early stopping after the warm-up period.
        # The retained model parameters are updated only when validation loss
        # improves by more than the predefined threshold.
        if epoch + 1 > WARMUP_EPOCHS:
            if val_loss_eval < retained_val_loss - MIN_DELTA:
                retained_val_loss = val_loss_eval
                retained_epoch = epoch + 1
                wait = 0

                retained_model_state = copy.deepcopy(model.state_dict())
                torch.save(retained_model_state, MODEL_PARAM_SAVE_PATH)

            else:
                wait += 1

            if wait >= PATIENCE:
                print(
                    "Early stopping at epoch %d. Retained epoch = %d, validation loss = %.6f"
                    % (epoch + 1, retained_epoch, retained_val_loss)
                )
                break

    # If no checkpoint was updated after the warm-up period, save the final state.
    # This prevents failure in unusual cases where the validation-loss criterion
    # is not triggered.
    if retained_model_state is not None:
        model.load_state_dict(retained_model_state)
    else:
        retained_epoch = epoch + 1
        retained_val_loss = val_loss_eval
        retained_model_state = copy.deepcopy(model.state_dict())
        torch.save(retained_model_state, MODEL_PARAM_SAVE_PATH)

    return {
        "model": model,
        "retained_model_state": retained_model_state,
        "retained_epoch": retained_epoch,
        "retained_val_loss": retained_val_loss,
        "train_loss_history": train_loss_history,
        "val_loss_history": val_loss_history
    }

In [ ]:
# ============================================================
# 8. Train DNN model
# ============================================================

output = train_with_validation_early_stopping(
    feature_train=feature_train,
    label_train=label_train,
    feature_val=feature_val,
    label_val=label_val,
    seed=MODEL_SEED
)

model = output["model"]

print("\nRetained epoch:", output["retained_epoch"])
print("Retained validation loss:", output["retained_val_loss"])
print("Final DNN parameters saved to:", MODEL_PARAM_SAVE_PATH)

In [ ]:
# ============================================================
# 9. Evaluate training and validation datasets
# ============================================================

train_metrics, train_pred = evaluate_internal_dataset(
    model,
    feature_train,
    label_train
)

val_metrics, val_pred = evaluate_internal_dataset(
    model,
    feature_val,
    label_val
)

print("\nInternal dataset performance:")
print_metric_summary("Train", train_metrics)
print_metric_summary("Validation", val_metrics)

internal_results_df = pd.DataFrame([
    {"Dataset": "Train", **train_metrics},
    {"Dataset": "Validation", **val_metrics}
])

print("\nInternal results:")
display(internal_results_df)

In [ ]:
# ============================================================
# 10. Evaluate 15 external verification experiments
# ============================================================

verification_results_df, verification_summary_df, pred_df_raw = evaluate_15_verification_experiments(
    model=model,
    verification_features=verification_features,
    true_df=true_df
)

print("\nDNN evaluation results for the 15 external verification experiments:")
print("Metrics are calculated using raw DNN predictions.")
print("No smoothing, first-point correction, normalization, or other post-processing is applied.")

print("\nExternal verification results:")
display(verification_results_df)

print("\nSummary of external verification results:")
display(verification_summary_df)

In [ ]:
# ============================================================
# 11. Training history
# ============================================================

training_history_df = pd.DataFrame({
    "Epoch": np.arange(1, len(output["train_loss_history"]) + 1),
    "Train_loss": output["train_loss_history"],
    "Validation_loss": output["val_loss_history"]
})

print("\nTraining history:")
display(training_history_df)

In [ ]:
# ============================================================
# 12. Summary plot for R2 and RMSE
# ============================================================

external_mean_r2 = verification_summary_df.loc[
    verification_summary_df["Metric"] == "R2", "Mean"
].values[0]

external_mean_rmse = verification_summary_df.loc[
    verification_summary_df["Metric"] == "RMSE", "Mean"
].values[0]

summary_plot_df = pd.DataFrame({
    "Dataset": [
        "Train",
        "Validation",
        "External 15"
    ],
    "R2": [
        train_metrics["R2"],
        val_metrics["R2"],
        external_mean_r2
    ],
    "RMSE": [
        train_metrics["RMSE"],
        val_metrics["RMSE"],
        external_mean_rmse
    ]
})

print("\nSummary plot data:")
display(summary_plot_df)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.5))

axes[0].bar(
    summary_plot_df["Dataset"],
    summary_plot_df["R2"]
)
axes[0].set_ylabel("R²")
axes[0].set_title("R² comparison")
axes[0].set_ylim(0, 1.05)

for i, v in enumerate(summary_plot_df["R2"]):
    axes[0].text(
        i,
        v + 0.015,
        "%.3f" % v,
        ha="center",
        va="bottom",
        fontsize=9
    )

axes[1].bar(
    summary_plot_df["Dataset"],
    summary_plot_df["RMSE"]
)
axes[1].set_ylabel("RMSE")
axes[1].set_title("RMSE comparison")

for i, v in enumerate(summary_plot_df["RMSE"]):
    axes[1].text(
        i,
        v,
        "%.2f" % v,
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.tight_layout()
plt.show()

print("\nWorkflow completed successfully.")